# Day 2 workspace restore (`d2_rag_eval`)
Rebuilds the Day 2 pipeline in any account and schema from this repo (`results/<date>/`) and the media files, with no AI calls except building the Cortex Search index. Run it top to bottom.

**Before you start (in your own account):**
1. Clone the repo into a Snowflake workspace (or upload it).
2. Put the media files in a stage, laid out as `<stage>/<file>` (`LESSON_SLIDES/`, `SLIDE_IMAGES/`, `LESSON_VIDEOS/`). For example, unzip `lab_export_d2_rag_eval_media_<date>.zip` and upload with the Snowflake CLI: `snow stage copy ./media/ @my_db.my_schema.restore_media/ --recursive`. (Snowsight uploads are limited in size; the videos are about 270 MB each.)
3. Optional, for fetching new lessons later: create the prerequisites in `results/<date>/prerequisites.md` (secret with your key, network rule, external access integration).

Then set the values in **Settings** and run all cells. The last cell checks that the runbook plan skips every step.

In [ ]:
# Where to restore: your own account's values. Replace every <...>.
TARGET_ROLE = "<YOUR_ROLE>"
TARGET_DATABASE = "<YOUR_DATABASE>"
TARGET_SCHEMA = "<YOUR_SCHEMA>"          # created if missing; must be empty
TARGET_WAREHOUSE = "<YOUR_WAREHOUSE>"

RESULTS_DATE = "2026-11-26"              # which results/<date>/ folder to restore from
MEDIA_SOURCE = "@<YOUR_DATABASE>.<YOUR_SCHEMA>.<MEDIA_STAGE>"   # folder holding LESSON_SLIDES/, SLIDE_IMAGES/, LESSON_VIDEOS/
SOURCE_DATABASE, SOURCE_SCHEMA = "BOOTCAMP", "ATP524"   # where the export came from (DDL that names it is pointed at the target)

WRITE_CONFIG = True         # writes the four TARGET_ values into sql/runbook_config.sql (and the 07a warehouse) so the runbook uses them
BUILD_SEARCH_SERVICE = True # builds course_search (indexing credits; it then refreshes daily and bills serving while it exists)

# Test restore inside the bootcamp account (tested 2026-11-26; leaves the files unchanged):
#   TARGET_ROLE, TARGET_DATABASE, TARGET_SCHEMA, TARGET_WAREHOUSE = "ALL_USERS_ROLE", "BOOTCAMP", "ATP524_RESTORE_TEST", "COMPUTE_WH"
#   MEDIA_SOURCE = "@BOOTCAMP.ATP524.LAB_EXPORT_STAGE_D2_RAG_EVAL/2026-11-26/stage_files"
#   WRITE_CONFIG = False      (then DROP SCHEMA BOOTCAMP.ATP524_RESTORE_TEST when done)

unset = [k for k, v in dict(TARGET_ROLE=TARGET_ROLE, TARGET_DATABASE=TARGET_DATABASE, TARGET_SCHEMA=TARGET_SCHEMA,
                            TARGET_WAREHOUSE=TARGET_WAREHOUSE, MEDIA_SOURCE=MEDIA_SOURCE).items() if "<" in v]
if unset:
    raise ValueError(f"Set these first: {', '.join(unset)}")

In [ ]:
import importlib, json, sys, time
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from snowflake.snowpark.context import get_active_session

WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "runbook_config.sql").exists())
sys.path.insert(0, str(WORKSPACE / "lab_metadata"))
import runbook_lib, export_lib
importlib.reload(runbook_lib); importlib.reload(export_lib)

RES = WORKSPACE / "results" / RESULTS_DATE
TYPES = json.loads((RES / "ddl" / "column_types.json").read_text())
MANIFEST = pd.read_csv(RES / "stage_manifest.csv")
T = f"{TARGET_DATABASE}.{TARGET_SCHEMA}"

session = get_active_session()
session.sql(f"USE ROLE {TARGET_ROLE}").collect()
session.sql(f"USE WAREHOUSE {TARGET_WAREHOUSE}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {T}").collect()
session.sql(f"USE SCHEMA {T}").collect()
existing = session.sql(f"SELECT COUNT(*) FROM {TARGET_DATABASE}.information_schema.tables WHERE table_schema = '{TARGET_SCHEMA.upper()}'").collect()[0][0]
if existing:
    raise RuntimeError(f"{T} already holds {existing} tables or views. Restore into an empty schema (or drop it first).")

if WRITE_CONFIG:
    export_lib.write_config(WORKSPACE, TARGET_ROLE, TARGET_DATABASE, TARGET_WAREHOUSE, TARGET_SCHEMA)
OPTIONS = {"MODE": "maintenance", "LABEL": "RESTORED", "EXPERIMENT_NAME": f"restored_{RESULTS_DATE}",
           "CHANGE_NOTE": f"Restored from export results/{RESULTS_DATE} (workspace_restore.ipynb)",
           "DRY_RUN": False, "FORCE_STEPS": [], "REQUIRE_APPROVAL_FOR_COST": False, "CONTINUE_ON_CHECK_FAIL": True,
           "INCLUDE_VIDEO": True, "INCLUDE_ANSWER_EVAL": True, "PROBE_CATALOG": False, "REFRESH_SOURCES": False}
rb = (runbook_lib.Runbook(session, WORKSPACE, OPTIONS) if WRITE_CONFIG else
      export_lib.runbook_for(runbook_lib, session, WORKSPACE, OPTIONS, TARGET_ROLE, TARGET_DATABASE, TARGET_WAREHOUSE, TARGET_SCHEMA))
rb.apply_config()   # session values ($eval_top_k, models, ...) and USE ... for the target
display(Markdown(f"Restoring `results/{RESULTS_DATE}/` into **`{T}`**: {sum(t['kind'] == 'table' for t in TYPES.values())} tables, "
                 f"{sum(t['kind'] == 'view' for t in TYPES.values())} views, {len(MANIFEST)} media files from `{MEDIA_SOURCE}`"))

In [ ]:
def present(sql):
    try:
        return bool(session.sql(sql).collect())
    except Exception:
        return False
prereq = pd.DataFrame([
    {"object": "secret dataexpert_api_key", "present": present(f"SHOW SECRETS LIKE 'DATAEXPERT_API_KEY' IN SCHEMA {T}"),
     "needed_for": "ingest procedures (steps 02, 12) and new lessons"},
    {"object": "external access integration dataexpert_api_access", "present": present("SHOW INTEGRATIONS LIKE 'DATAEXPERT_API_ACCESS'"),
     "needed_for": "ingest procedures (steps 02, 12) and new lessons"},
    {"object": f"media in {MEDIA_SOURCE}", "present": present(f"LIST {MEDIA_SOURCE}/"), "needed_for": "slide parsing, images, video (required)"},
])
display(prereq)
if not prereq.iloc[2]["present"]:
    raise RuntimeError(f"No media found at {MEDIA_SOURCE}. Upload the media folders first (see the top of this notebook).")
if not prereq.iloc[:2]["present"].all():
    display(Markdown("Ingest prerequisites are missing: everything else restores, ingest procedures are skipped, and "
                     "`REFRESH_SOURCES=False` is used. See `prerequisites.md` to add them later, then rerun step 02 / 12."))

## Tables and data
Creates the stages and every Day 2 table with its exact DDL, then loads each CSV (and its parts) with the exact column types: JSON text back to VARIANT / ARRAY / OBJECT / VECTOR, timestamps with their time zone, `\N` back to NULL.

In [ ]:
for stmt in runbook_lib.split_statements((RES / "ddl" / "stages.sql").read_text()):
    session.sql(stmt).collect()
for stmt in runbook_lib.split_statements((RES / "ddl" / "tables.sql").read_text()):
    session.sql(export_lib.retarget(stmt, SOURCE_DATABASE, SOURCE_SCHEMA, TARGET_DATABASE, TARGET_SCHEMA)).collect()

loads = []
for name, t in TYPES.items():
    if t["kind"] != "table":
        continue
    n = sum(export_lib.load_csv(session, p, f"{T}.{export_lib.q(name)}", t["columns"])
            for p in export_lib.csv_paths(RES / "tables", name))
    got = session.sql(f"SELECT COUNT(*) FROM {T}.{export_lib.q(name)}").collect()[0][0]
    loads.append({"table": name, "csv_rows": n, "table_rows": got, "ok": n == got})
loads = pd.DataFrame(loads)
display(Markdown(f"Loaded {len(loads)} tables, {loads.csv_rows.sum()} rows; mismatches: {(~loads.ok).sum()}"))
display(loads[~loads.ok] if (~loads.ok).any() else loads.head(3))

In [ ]:
for stage in MANIFEST.stage.unique():
    session.sql(f"COPY FILES INTO @{T}.{stage}/ FROM {MEDIA_SOURCE}/{stage}/").collect()
    session.sql(f"ALTER STAGE {T}.{stage} REFRESH").collect()
restored = export_lib.stage_manifest(session, TARGET_DATABASE, TARGET_SCHEMA, list(MANIFEST.stage.unique()))
check = MANIFEST.merge(restored, on=["stage", "relative_path"], how="left", suffixes=("", "_restored"))
check["ok"] = (check["size"] == check["size_restored"]) & (check["md5"] == check["md5_restored"])
display(check.groupby("stage").agg(files=("ok", "size"), matched=("ok", "sum")).reset_index())
if not check.ok.all():
    raise RuntimeError(f"{(~check.ok).sum()} media files are missing or differ: " + ", ".join(check[~check.ok].relative_path.head(10)))

## Code objects
Views come from the exported DDL (pointed at this schema). Functions and procedures come from the repo's own SQL (`01`, `02`, `08`, `10`, `12`), so they always match the code you are running. Ingest procedures need the secret and integration; without them they are listed as skipped.

In [ ]:
views = runbook_lib.split_statements((RES / "ddl" / "views.sql").read_text())
pending, failed, errors = views, [], {}
for _ in range(len(views)):             # views that read other views are retried until all compile
    failed = []
    for stmt in pending:
        try:
            session.sql(export_lib.retarget(stmt, SOURCE_DATABASE, SOURCE_SCHEMA, TARGET_DATABASE, TARGET_SCHEMA)).collect()
        except Exception as exc:
            failed.append(stmt); errors[stmt[:60]] = str(exc)[:200]
    if not failed or len(failed) == len(pending):
        break
    pending = failed
code = [{"object": f"{len(views) - len(failed)} of {len(views)} views", "ok": not failed,
         "detail": "; ".join(errors[s[:60]] for s in failed)}]

# Functions from the exported DDL (01.build also makes them, but its ALTER ... ADD COLUMN IF NOT EXISTS
# fails with 'ambiguous column name' once the restored tables already have those columns)
for stmt in runbook_lib.split_statements((RES / "ddl" / "functions.sql").read_text()):
    try:
        session.sql(export_lib.retarget(stmt, SOURCE_DATABASE, SOURCE_SCHEMA, TARGET_DATABASE, TARGET_SCHEMA)).collect()
        code.append({"object": stmt.split("(")[0].split()[-1], "ok": True, "detail": "function from ddl/functions.sql"})
    except Exception as exc:
        code.append({"object": "function", "ok": False, "detail": str(exc)[:200]})
# Procedures from the repo's own SQL, so they match the code you run
for sid in ("02.define", "08.define", "10.define", "12.define"):
    try:
        rb.run_section(rb.sections[sid])
        code.append({"object": sid, "ok": True, "detail": rb.sections[sid].purpose})
    except Exception as exc:
        code.append({"object": sid, "ok": False, "detail": str(exc)[:200]})
code = pd.DataFrame(code)
display(code)

In [ ]:
if BUILD_SEARCH_SERVICE:
    import re
    sql = re.sub(r"WAREHOUSE = '[^']+'", f"WAREHOUSE = '{TARGET_WAREHOUSE.upper()}'", rb.sections["07a.build"].sql, count=1)
    for stmt in runbook_lib.split_statements(sql):
        session.sql(stmt).collect()
    for _ in range(60):
        svc = [r.as_dict() for r in session.sql(f"SHOW CORTEX SEARCH SERVICES LIKE '{rb.config['search_service']}' IN SCHEMA {T}").collect()][0]
        if svc["serving_state"] == "ACTIVE" and svc["indexing_state"] == "ACTIVE":
            break
        time.sleep(10)
    display(Markdown(f"`{svc['name']}`: serving {svc['serving_state']}, indexing {svc['indexing_state']}, {svc['source_data_num_rows']} rows"))
else:
    display(Markdown("BUILD_SEARCH_SERVICE = False: the runbook builds it (step 07a) on its next run."))

## Adopt the restored state
Records every step's current fingerprints as one restored run, scores it, compares it with the exported champion (expect all ties), and promotes it. Without this, the first runbook run would read small CSV round-trip differences as changed inputs and rerun the AI steps.

In [ ]:
rb.ensure_record_tables()
rb.start_run()
adopted = rb.adopt_restored_state(f"restored from export results/{RESULTS_DATE}")
plan = rb.make_plan()
scores = rb.record_scores()
comparisons = rb.default_comparisons()
rb.end_run("SUCCESS", {"restored_from": RESULTS_DATE, "adopted": int(adopted.adopted.sum())})
reason = f"restored from export results/{RESULTS_DATE}; was champion {rb.champion_run_id}"
display(Markdown(rb.promote(rb.run_id, reason)))
display(Markdown(f"Adopted {int(adopted.adopted.sum())} of {len(adopted)} steps"))
display(adopted[~adopted.adopted | adopted.detail.str.contains("FAILED")])
for result, _, _ in comparisons:
    display(Markdown(f"- {result['summary']}"))
for v in [v for v in rb.verdicts if v.get("verdict") == "not comparable"]:
    display(Markdown(f"- {v['summary']}"))

In [ ]:
check_rb = (runbook_lib.Runbook(session, WORKSPACE, OPTIONS) if WRITE_CONFIG else
            export_lib.runbook_for(runbook_lib, session, WORKSPACE, OPTIONS, TARGET_ROLE, TARGET_DATABASE, TARGET_WAREHOUSE, TARGET_SCHEMA))
check_rb.apply_config()
final_plan = check_rb.make_plan()
check_rb.halted_reason = None
would_run = final_plan[final_plan.decision.isin(["RUN", "DRIFT"])]
display(Markdown(f"Plan: {len(final_plan)} steps, {len(would_run)} would run"))
display(would_run if len(would_run) else final_plan.head(3))
display(session.sql(f"""SELECT method, questions, hit_at_1, mrr_at_10, recall_at_10, ndcg_at_10
                        FROM {T}.eval_run_report WHERE run_id = '{rb.run_id}' AND slice_kind = 'all' ORDER BY method""").to_pandas())